# Importing and cleaning datasets for final visualizations

In [3]:
import pandas as pd
import re
import pycountry
import matplotlib.pyplot as plt

## "Dwarfing" Fish Production/Landings Graph 

Data Source: FIRMS Global Tuna Atlas (https://www.fao.org/fishery/en/collection/firms-tuna-atlas)

Acronym: GTA

metadata here: https://www.fao.org/fishery/geonetwork/srv/eng/catalog.search#/metadata/global_nominal_catch_firms_level0



Catches of Bluefin species + Bigeye and Yellofin tuna up to 2025

In [ ]:
#import cell is seperate so reach running of notebook starts from the same place (data un-altered)
GTA_all = pd.read_csv("datasets/global_nominal_catch_firms_level0_harmonized.csv")
GTA_all.head()


In [ ]:
GTA_all.dtypes
GTA_all["measurement_type"].value_counts()


In [ ]:
#first four numbers in time_start is the year -> extract and make new column
GTA_all['year'] = GTA_all['time_start'].str[:4]
#remove:
    #gear_type - cannot reasonably assume all tuna caught with certain gear type are used for the same purpose (fishing vs ranching)
    #fishing_mode
    #measurement_processing_level
    #measurement_unit - the same for all (it's tonnes)
    #source_authority
    #geographic_identifier
    #measurement
GTA_all = GTA_all.drop(columns=["measurement",'gear_type', 'fishing_mode', 'measurement_processing_level', 'measurement_unit', 'source_authority', 'geographic_identifier'])
#remove rows with measurement_type DD or DL or NL (Nominal Landings, which is a subset of NC Nominal Catch)
GTA_all = GTA_all[~GTA_all['measurement_type'].isin(['DD', 'DL', 'NL'])]
#and then drop the measurement_type column since we are only looking at one type of measurement (NC)
GTA_all = GTA_all.drop(columns=['measurement_type'])
#drop time start, time end
GTA_all = GTA_all.drop(columns=['time_start', 'time_end'])


#keep the following fish: 
    #BFT Atlantic bluefin tuna
    #BET Bigeye tuna
    #PBF Pacific bluefin tuna
    #SBF Southern bluefin tuna
    #YFT Yellowfin tuna
tuna_keep = ['BFT', 'BET', 'PBF', 'SBF', 'YFT']


GTA_tuna = GTA_all[GTA_all['species'].isin(tuna_keep)]
GTA_tuna_grouped = GTA_tuna.groupby(['species', 'year'])['measurement_value'].sum().reset_index()


### Export GTA FIRMS

In [ ]:
## EXPORT
GTA_tuna.to_csv("datasets/export/GTA_FIRMs_tuna_cleaned_countries.csv", index=False)
GTA_tuna_grouped.to_csv("datasets/export/GTA_FIRMs_tuna_cleaned_grouped.csv", index=False)


In [ ]:
GTA_tuna.head(10)

In [ ]:
#group by species and year

GTA_tuna_grouped.head(10)


#graph only bluefins, measurement_value over time
GTA_bluefins = GTA_tuna_grouped[GTA_tuna_grouped['species'].isin(['BFT', 'PBF', 'SBF'])]
plt.figure(figsize=(10,6))
for species in ['BFT', 'PBF', 'SBF']:
    species_data = GTA_bluefins[GTA_bluefins['species'] == species]
    plt.plot(species_data['year'], species_data['measurement_value'], label=species)
plt.xlabel('Year')
plt.ylabel('Measurement Value (tonnes)')
plt.title('Global Nominal Catch of Bluefin Tuna Over Time')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#same but stacked barplot
GTA_bluefins_pivot = GTA_bluefins.pivot(index='year', columns='species', values='measurement_value').fillna(0)
GTA_bluefins_pivot.plot(kind='bar', stacked=True, figsize=(10,6))
plt.xlabel('Year')
plt.ylabel('Measurement Value (tonnes)')
plt.title('Global Nominal Catch of Bluefin Tuna Over Time (Stacked)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#stacked barplot of all tuna species
GTA_tuna_pivot = GTA_tuna_grouped.pivot(index='year', columns='species', values='measurement_value').fillna(0)
GTA_tuna_pivot.plot(kind='bar', stacked=True, figsize=(10,6))
plt.xlabel('Year')
plt.ylabel('Measurement Value (tonnes)')
plt.title('Global Nominal Catch of Tuna Species Over Time (Stacked)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Map - 5° Lat-Long of Global Tuna Catches 

Source: GTA FIRMS 


will call this object "tuna_map"

In [ ]:
tuna_map = pd.read_csv("datasets/tuna_map/global_catch_5deg_1m_firms_level0_harmonized.csv")
tuna_map.head()

In [ ]:
tuna_map["measurement_unit"].value_counts()

In [ ]:
#this is a pretty big dataset that includes all tuna and shark like species, so will treat it similarly to the previous dataset.


#create "year" column from time_start
tuna_map['year'] = tuna_map['time_start'].str[:4]

#only retain species of interest
tuna_keep = ['BFT', 'BET', 'PBF', 'SBF', 'YFT']
tuna_map_soi = tuna_map[tuna_map['species'].isin(tuna_keep)]

#remove rows with measurement_type DD or DL
tuna_map_soi = tuna_map_soi[~tuna_map_soi['measurement_type'].isin(['DD', 'DL'])]

#remove:
    #gear_type - cannot reasonably assume all tuna caught with certain gear type are used for the same purpose (fishing vs ranching)
    #fishing_mode
    #measurement_processing_level
    #source_authority
    #measurement
    #time_start
    #time_end
tuna_map_soi = tuna_map_soi.drop(columns=['gear_type', 'fishing_mode', 'measurement_processing_level', 'source_authority', 'measurement', 'measurement_type', 'time_start', 'time_end'])



In [ ]:
tuna_map_soi["measurement_unit"].value_counts()

In [ ]:
#map with only tuna of interst, only neccesary columns
tuna_map_soi.head()

#need to figure out what to do with count vs tonne data
tuna_map_soi_tonne = tuna_map_soi[tuna_map_soi['measurement_unit'] == 't']
tuna_map_soi_count = tuna_map_soi[tuna_map_soi['measurement_unit'] == 'no']


In [ ]:
#group by geographic identifier, year, and species, sum measurement value
tuna_map_soi_count_grouped = tuna_map_soi_count.groupby(['geographic_identifier', 'year', 'species'])['measurement_value'].sum().reset_index()
tuna_map_soi_tonne_grouped = tuna_map_soi_tonne.groupby(['geographic_identifier', 'year', 'species'])['measurement_value'].sum().reset_index()
tuna_map_soi_count_grouped.head()

In [ ]:
#what is the first year of data for each species?
tuna_map_soi_count_grouped.groupby('species')['year'].min()

#remove years before 1965 for even coverage across species
tuna_map_soi_count_grouped = tuna_map_soi_count_grouped[tuna_map_soi_count_grouped['year'] >= '1965']
tuna_map_soi_tonne_grouped = tuna_map_soi_tonne_grouped[tuna_map_soi_tonne_grouped['year'] >= '1965']
tuna_map_soi_tonne_grouped.head()

### EXPORT tuna dataset for map

In [ ]:
#export tonne and count
tuna_map_soi_count_grouped.to_csv("datasets/export/tuna_map_count.csv")
tuna_map_soi_tonne_grouped.to_csv("datasets/export/tuna_map_tonne.csv")


### Geographic Key for tuna map

Source: https://www.fao.org/fishery/geonetwork/srv/eng/catalog.search#/metadata/cwp-grid-map-5deg_x_5deg

Key for 5° by 5° grid overlaid on earth

In [ ]:
map_geographic_key = pd.read_csv("datasets/tuna_map/cwp-grid-map-5deg_x_5deg.csv")
map_geographic_key.head()

how to transform this into something mapbox can read?

In [ ]:
"""
build_catch_geojson.py
----------------------
Merges tuna catch CSVs (count + tonne) into the CWP 5° grid GeoJSON
using wide format (one column per year).

Inputs:
  - cwp-grid-5deg.geojson       (corrected grid from previous step)
  - tuna_map_count.csv
  - tuna_map_tonne.csv

Output:
  - cwp-grid-5deg-catch.geojson

Usage:
  python build_catch_geojson.py
"""

import pandas as pd
import json
import os

# ── Paths (edit these if needed) ─────────────────────────────────────────────
GRID_GEOJSON  = 'datasets/tuna_map/cwp-grid-5deg-var3.geojson'
COUNT_CSV     = 'datasets/export/tuna_map_count.csv'
TONNE_CSV     = 'datasets/export/tuna_map_tonne.csv'
OUTPUT_GEOJSON = 'datasets/export/cwp-grid-5deg-catch.geojson'

# ── 1. Load and pivot catch data to wide format ───────────────────────────────
print("Loading catch data...")
count_df = pd.read_csv(COUNT_CSV)
tonne_df = pd.read_csv(TONNE_CSV)

def to_wide(df, prefix):
    """Aggregate all species per square per year, then pivot to wide."""
    return (df
        .groupby(['geographic_identifier', 'year'])['measurement_value']
        .sum()
        .reset_index()
        .pivot(index='geographic_identifier', columns='year', values='measurement_value')
        .rename(columns=lambda y: f'{prefix}_{y}'))

count_wide = to_wide(count_df, 'count')
tonne_wide = to_wide(tonne_df, 'tonne')

catch_wide = count_wide.join(tonne_wide, how='outer')
catch_wide.index.name = 'CWP_CODE'
catch_wide = catch_wide.reset_index()

print(f"  Wide table: {catch_wide.shape[0]} squares × {catch_wide.shape[1]} columns")

# ── 2. Load GeoJSON ───────────────────────────────────────────────────────────
print(f"Loading {GRID_GEOJSON}...")
with open(GRID_GEOJSON) as f:
    geojson = json.load(f)

# ── 3. Merge catch data into GeoJSON properties ───────────────────────────────
print("Merging...")
catch_lookup = catch_wide.set_index('CWP_CODE').to_dict(orient='index')

matched, unmatched = 0, 0
for feature in geojson['features']:
    code = feature['properties']['CWP_CODE']
    if code in catch_lookup:
        for k, v in catch_lookup[code].items():
            feature['properties'][k] = (
                None if (isinstance(v, float) and pd.isna(v))
                else round(float(v), 3)
            )
        matched += 1
    else:
        unmatched += 1

print(f"  Matched: {matched} | No catch data: {unmatched}")

# ── 4. Write output ───────────────────────────────────────────────────────────
print(f"Writing {OUTPUT_GEOJSON}...")
with open(OUTPUT_GEOJSON, 'w') as f:
    json.dump(geojson, f)

size_mb = os.path.getsize(OUTPUT_GEOJSON) / 1e6
print(f"Done. Output: {OUTPUT_GEOJSON} ({size_mb:.1f} MB)")

## NOAA Import/Export

https://www.fisheries.noaa.gov/foss/f?p=215:2:14285563639484:::::

In [4]:
fish_import_export = pd.read_csv("datasets/noaa_import_export_tunas.csv")
#column names are in first row, so set header to 0 and skip the first row
fish_import_export.columns = fish_import_export.iloc[0]
fish_import_export = fish_import_export[1:]
fish_import_export.head()


,Year,Source,Product Name,Country Name,Volume (kg),Value (USD),Calculated Duty (USD),Edible code
1,1994,IMP,TUNA BLUEFIN FRESH,UNITED KINGDOM,751,"54,423",0,E
2,1993,IMP,TUNA BLUEFIN FRESH,AUSTRALIA,"2,327","21,938",0,E
3,1996,IMP,TUNA BLUEFIN FRESH,FRANCE,"9,730","56,893",0,E
4,1999,IMP,TUNA BLUEFIN FRESH,CANADA,"90,313","658,114",0,E
5,2001,IMP,TUNA BLUEFIN FRESH,TAIWAN,997,"19,940",0,E


In [5]:
#year to int
fish_import_export['Year'] = fish_import_export['Year'].astype(int)
# value and volume have commas, so remove commas and convert to int
fish_import_export['Value (USD)'] = fish_import_export['Value (USD)'].str.replace(',', '').astype(int)
fish_import_export['Volume (kg)'] = fish_import_export['Volume (kg)'].str.replace(',', '').astype(int)
#drop calculated duty and edible code
fish_import_export = fish_import_export.drop(columns=['Calculated Duty (USD)', 'Edible code'])




In [6]:
fish_import_export.dtypes
fish_import_export.head()


,Year,Source,Product Name,Country Name,Volume (kg),Value (USD)
1,1994,IMP,TUNA BLUEFIN FRESH,UNITED KINGDOM,751,54423
2,1993,IMP,TUNA BLUEFIN FRESH,AUSTRALIA,2327,21938
3,1996,IMP,TUNA BLUEFIN FRESH,FRANCE,9730,56893
4,1999,IMP,TUNA BLUEFIN FRESH,CANADA,90313,658114
5,2001,IMP,TUNA BLUEFIN FRESH,TAIWAN,997,19940


In [7]:
#get rid of:
    #anything that includes the word EVISCERATED
    #OTHER PREPARATIONS
keywords_to_remove = ['EVISCERATED', 'OTHER PREPARATIONS'] #use these keywords to filter
fish_import_export = fish_import_export[~fish_import_export["Product Name"].str.contains('|'.join(keywords_to_remove))]

#All Product Name containing TUNA BLUEFIN become TUNA BLUEFIN 
fish_import_export["Product Name"] = fish_import_export["Product Name"].str.replace(r'TUNA BLUEFIN.*', 'TUNA BLUEFIN', regex=True)
#same for YELLOWFIN and BIGEYE
fish_import_export["Product Name"] = fish_import_export["Product Name"].str.replace(r'TUNA YELLOWFIN.*', 'TUNA YELLOWFIN', regex=True)
fish_import_export["Product Name"] = fish_import_export["Product Name"].str.replace(r'TUNA BIGEYE.*', 'TUNA BIGEYE', regex=True)


fish_import_export["Product Name"].value_counts()


Product Name
TUNA YELLOWFIN    6857
TUNA BIGEYE       2680
TUNA BLUEFIN      2509
Name: count, dtype: int64

In [10]:
#also prepare bluefin import only subsection
fish_import_bluefin = fish_import_export[fish_import_export["Product Name"].str.contains("TUNA BLUEFIN")]
fish_import_bluefin = fish_import_bluefin[fish_import_bluefin["Source"] == "IMP"]
fish_import_bluefin.head()

,Year,Source,Product Name,Country Name,Volume (kg),Value (USD)
1,1994,IMP,TUNA BLUEFIN,UNITED KINGDOM,751,54423
2,1993,IMP,TUNA BLUEFIN,AUSTRALIA,2327,21938
3,1996,IMP,TUNA BLUEFIN,FRANCE,9730,56893
4,1999,IMP,TUNA BLUEFIN,CANADA,90313,658114
5,2001,IMP,TUNA BLUEFIN,TAIWAN,997,19940


### Export NOAA imp/exp

In [13]:
fish_import_export.to_csv("datasets/export/noaa_import_export_tunas_combined_cleaned.csv", index=False)
fish_import_bluefin.to_csv("datasets/export/noaa_import_tunas_bluefin.csv", index=False)